In [1]:
import sys
print(sys.executable)

/home/user2/Desktop/Rag_pipeline_to-check_compliance/.venv/bin/python


In [2]:
import json
from typing import List
import os
import base64
import uuid
import json
from pathlib import Path
from typing import List
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langchain_core.documents import Document

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
# from langchain_ollama import OllamaLLM
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from unstructured.documents.elements import Table,Image
from concurrent.futures import ThreadPoolExecutor, TimeoutError


/home/user2/Desktop/Rag_pipeline_to-check_compliance/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    print(f"✅ Extracted {len(elements)} elements")
    

    
    return elements

# Test with your PDF file
file_path = "./annual_report_docs//dummy_annual_report.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

📄 Partitioning document: ./annual_report_docs//dummy_annual_report.pdf


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 696.15it/s]
The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


✅ Extracted 4 elements


In [4]:
elements

In [5]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [6]:
elements[2].to_dict()

{'type': 'Table',
 'element_id': '7aa3c5685849f7bd1a16a8e2bd52109b',
 'text': 'Category Amount Revenue n500,000 Expenses n300,000 Net Profit n200,000',
 'metadata': {'detection_class_prob': 0.7467113137245178,
  'coordinates': {'points': ((np.float64(323.85400390625),
     np.float64(492.0706787109375)),
    (np.float64(323.85400390625), np.float64(680.901123046875)),
    (np.float64(1064.054443359375), np.float64(680.901123046875)),
    (np.float64(1064.054443359375), np.float64(492.0706787109375))),
   'system': 'PixelSpace',
   'layout_width': 1654,
   'layout_height': 2339},
  'last_modified': '2026-03-13T09:51:32',
  'text_as_html': '<table><tbody><tr><td>Category</td><td>Amount</td></tr><tr><td>Revenue</td><td>500,000</td></tr><tr><td>Expenses</td><td>300,000</td></tr><tr><td>Net Profit</td><td>200,000</td></tr></tbody></table>',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': './annual_report_docs',
  'filename': 'dummy_annual_rep

In [ ]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

In [7]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html 


Found 1 tables


{'type': 'Table',
 'element_id': '7aa3c5685849f7bd1a16a8e2bd52109b',
 'text': 'Category Amount Revenue n500,000 Expenses n300,000 Net Profit n200,000',
 'metadata': {'detection_class_prob': 0.7467113137245178,
  'coordinates': {'points': ((np.float64(323.85400390625),
     np.float64(492.0706787109375)),
    (np.float64(323.85400390625), np.float64(680.901123046875)),
    (np.float64(1064.054443359375), np.float64(680.901123046875)),
    (np.float64(1064.054443359375), np.float64(492.0706787109375))),
   'system': 'PixelSpace',
   'layout_width': 1654,
   'layout_height': 2339},
  'last_modified': '2026-03-13T09:51:32',
  'text_as_html': '<table><tbody><tr><td>Category</td><td>Amount</td></tr><tr><td>Revenue</td><td>500,000</td></tr><tr><td>Expenses</td><td>300,000</td></tr><tr><td>Net Profit</td><td>200,000</td></tr></tbody></table>',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': './annual_report_docs',
  'filename': 'dummy_annual_rep

In [12]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=10 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 2 chunks


In [13]:


def separate_content_types(chunk):
    """Analyze what types of content are in a chunk and save images to folder"""
    
    # Create images folder if it doesn't exist
    images_folder = "extracted_images"
    Path(images_folder).mkdir(parents=True, exist_ok=True)
    
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],  # Will store file paths
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    
                    # Save image to extracted_images folder
                    img_data = element.metadata.image_base64
                    img_filename = f"image_{uuid.uuid4()}.jpg"
                    img_path = os.path.join(images_folder, img_filename)
                    
                    # Decode and save
                    with open(img_path, "wb") as f:
                        f.write(base64.b64decode(img_data))
                    
                    # Store the file path
                    content_data['images'].append(img_path)
                    print(f"     → Saved image to: {img_path}")
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

def summarize_images(images: List[str]) -> str:
    """Use Qwen model to summarize images"""
    if not images:
        return ""
    
    try:
        print(f"     → Using Qwen3.5 for {len(images)} image(s)")
        llm = ChatOllama(model="qwen3.5:0.8b", temperature=0)
        
        # message_content = [{"type": "text", "text": "Describe what you see in this image in detail:"}]
        message_content = [
                            {
                                "type": "text",
                                "text": """Convert this diagram into knowledge for a retrieval system.

                        Extract:
                        - Main topic
                        - Key concepts
                        - Conditions or rules shown

                        Write a concise explanation that preserves the meaning of the diagram.

                        Do not describe visual appearance.
                        Do not infer information not present in the diagram.
                        Limit output to 120 words."""
                            }
                        ]
        # Add images
        for image_path in images:
            message_content.append({
                "type": "image_url",
                "image_url": image_path
            })
        
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        print("DEBUG IMAGE RESPONSE:", repr(response.content))
        
        return response.content
    except Exception as e:
        print(f"     ❌ Qwen image summary failed: {e}")
        return f"[{len(images)} image(s) - summary failed]"

def summarize_tables(tables: List[str]) -> str:
    """Use Llama model to summarize tables"""
    if not tables:
        return ""
    
    try:
        print(f"     → Using Llama3.2 for {len(tables)} table(s)")
        llm = ChatOllama(model="llama3.2:1b", temperature=0)
        
        prompt_text = "The following tables were extracted from a document.\n\n"
        for i, table in enumerate(tables):
            prompt_text += f"Table {i+1}:\n{table}\n\n"
        

        prompt_text += """
                        YOUR TASK:
                        Generate a comprehensive, searchable description that covers:

                        1. Key facts, numbers, and data points from text and tables
                        2. Main topics and concepts discussed
                        3. Questions this content could answer
                        4. Alternative search terms users might use

                        Make it detailed and searchable - prioritize findability over brevity.

                        SEARCHABLE DESCRIPTION:
                        """

        
        # message = HumanMessage(content=prompt_text)
        
        response = llm.invoke([prompt_text])
        
        return response.content
    except Exception as e:
        print(f"     ❌ Llama table summary failed: {e}")
        return f"[{len(tables)} table(s) - summary failed]"

def create_ai_enhanced_summary(tables: List[str], images: List[str]) -> str:
    """Create summaries for tables (Llama) and images (Qwen) using different models"""
    
    summaries = []
    
    # Summarize tables with Llama
    if tables:
        table_summary = summarize_tables(tables)
        if table_summary:
            summaries.append(f"TABLE SUMMARIES:\n{table_summary}")
    
    # Summarize images with Qwen
    if images:
        image_summary = summarize_images(images)
        if image_summary:
            summaries.append(f"IMAGE DESCRIPTIONS:\n{image_summary}")
    
    return "\n\n".join(summaries)

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries using different models"""
    print("🧠 Processing chunks with AI Summaries...")
    print("   - Images: Qwen3.5:0.8b")
    print("   - Tables: Llama3.2:1b")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"\n   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Start with raw text
        final_content = content_data['text']
        
        # Create summaries for tables/images using appropriate models
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summaries...")
            try:
                # Send tables to Llama, images to Qwen
                ai_summary = create_ai_enhanced_summary(
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     ✓ AI summaries created successfully")
                
                # Combine raw text + AI summaries
                if ai_summary:
                    final_content = f"""{content_data['text']}

--- TABLES AND IMAGES SUMMARY ---
{ai_summary}"""
                
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                # Keep raw text only if AI fails
                final_content = content_data['text']
        else:
            print(f"     → No tables/images, using raw text only")
        
        # Create LangChain Document
        doc = Document(
            page_content=final_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                }),
                "chunk_number": current_chunk,
                "content_types": content_data['types']
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"\n✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents

# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   - Images: Qwen3.5:0.8b
   - Tables: Llama3.2:1b

   Processing chunk 1/2
     Types found: ['text']
     Tables: 0, Images: 0
     → No tables/images, using raw text only

   Processing chunk 2/2
     Types found: ['text', 'table']
     Tables: 1, Images: 0
     → Creating AI summaries...
     → Using Llama3.2 for 1 table(s)
     ✓ AI summaries created successfully

✅ Processed 2 chunks


In [14]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "ABC Company Annual Report 2024", "tables_html": [], "images_base64": []}', 'chunk_number': 1, 'content_types': ['text']}, page_content='ABC Company Annual Report 2024'),
 Document(metadata={'original_content': '{"raw_text": "Financial Summary\\n\\nCategory Amount Revenue n500,000 Expenses n300,000 Net Profit n200,000\\n\\nReport Date: 31-03-2024", "tables_html": ["<table><tbody><tr><td>Category</td><td>Amount</td></tr><tr><td>Revenue</td><td>500,000</td></tr><tr><td>Expenses</td><td>300,000</td></tr><tr><td>Net Profit</td><td>200,000</td></tr></tbody></table>"], "images_base64": []}', 'chunk_number': 2, 'content_types': ['text', 'table']}, page_content="Financial Summary\n\nCategory Amount Revenue n500,000 Expenses n300,000 Net Profit n200,000\n\nReport Date: 31-03-2024\n\n--- TABLES AND IMAGES SUMMARY ---\nTABLE SUMMARIES:\nBased on the provided tables, here is a comprehensive, searchable description that covers the key facts, mai

In [15]:
def export_chunks_to_json(chunks, filename="annual_chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 2 chunks to annual_chunks_export.json


In [21]:
from langchain_community.vectorstores.utils import filter_complex_metadata

def create_vector_store(documents, persist_directory="dbv2/chroma_db"):
    """Create and persist ChromaDB vector store"""
    
    print("🔮 Creating embeddings and storing in ChromaDB...")
        
    embedding_model = OllamaEmbeddings(model="nomic-embed-text")

    # ⭐ Fix: remove unsupported metadata types (lists, dicts)
    documents = filter_complex_metadata(documents)
    
    print("--- Creating vector store ---")
    
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    
    print("--- Finished creating vector store ---")
    print(f"✅ Vector store created and saved to {persist_directory}")
    
    return vectorstore


# Create the vector store
db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv2/chroma_db


In [25]:
# After your retrieval
query = "Name of company  that has financial report"
retriever = db.as_retriever(search_kwargs={"k": 2})
chunks = retriever.invoke(query)
# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

✅ Exported 2 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': "Financial Summary\n\nCategory Amount Revenue n500,000 Expenses n300,000 Net Profit n200,000\n\nReport Date: 31-03-2024\n\n--- TABLES AND IMAGES SUMMARY ---\nTABLE SUMMARIES:\nBased on the provided tables, here is a comprehensive, searchable description that covers the key facts, main topics, questions, and alternative search terms:\n\n**Key Facts and Data Points:**\n\n* Revenue: $500,000\n* Expenses: $300,000\n* Net Profit: $200,000\n\n**Main Topics and Concepts:**\n\n* Financial statements (Revenue, Expenses, Net Profit)\n* Accounting principles (Assets, Liabilities, Equity)\n* Business operations (Revenue, Expenses, Net Profit)\n\n**Questions this Content Could Answer:**\n\n* What are the financial performance metrics of the business?\n* What are the key drivers of the business's financial success?\n* How do the financial statements relate to the business's overall operations?\n* What are the implications of the business's financial performance

In [ ]:
query = "Explain whole financial summary of ABC company"

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in chunks])

llm = ChatOllama(model="llama3.2:1b", temperature=0)

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer clearly and concisely.
"""

response = llm.invoke(prompt)

print(response.content)

Here is a clear and concise explanation of the whole financial summary of ABC Company:

ABC Company's financial summary for the year 2024 is presented in the following tables:

**Revenue:** $500,000
**Expenses:** $300,000
**Net Profit:** $200,000

These figures indicate that the company generated $500,000 in revenue, incurred $300,000 in expenses, and achieved a net profit of $200,000. This suggests that the company was able to generate sufficient revenue to cover its expenses and achieve a profit.

The financial statements also provide some key insights into the company's operations and performance. For example, the revenue has increased by 50% from the previous year, indicating that the company has been able to grow its business. The expenses have decreased by 33% from the previous year, suggesting that the company has been able to reduce its costs and improve its efficiency.

The net profit of $200,000 is a significant improvement from the previous year, indicating that the company 